# Cours 3 : Nettoyage, Transformation et Agrégation (2h)
## Exercices d'application en direct (FFP)

Ce notebook met en pratique les trois grands piliers vus en cours :
1. **Nettoyage raisonné** : gestion des `NaN`, suppression des doublons partiels et détection d'outliers par IQR.
2. **Réconciliation relationnelle** : jointures avec `pd.merge()` et audit avec `indicator=True`.
3. **Agrégations & Remodelage** : pattern *Split-Apply-Combine* (`groupby`, `agg`) et tableaux croisés (`pivot_table`).

In [ ]:
import pandas as pd
import numpy as np

print("Bibliothèques importées avec succès !")

## Partie 1 : Nettoyage raisonné (NaN, Doublons & Outliers IQR)

Exécutez la cellule suivante pour générer une table brute de transactions logistiques.

In [ ]:
np.random.seed(42)
n = 1000

df_tx = pd.DataFrame({
    "tx_id": [f"TX-{i:04d}" for i in range(1, n + 1)],
    "client_id": [f"CLI-{np.random.randint(100, 200)}" for _ in range(n)],
    "montant": np.random.normal(loc=100, scale=25, size=n).round(2),
    "statut": np.random.choice(["VALIDE", "EN_ATTENTE", None], size=n, p=[0.75, 0.20, 0.05])
})

# Injection de pièges
# 1. NaN sur montant
df_tx.loc[10:15, "montant"] = np.nan
# 2. Doublons partiels (même transaction répétée)
df_tx = pd.concat([df_tx, df_tx.iloc[20:25]], ignore_index=True)
# 3. Outliers extrêmes
df_tx.loc[3, "montant"] = 1500.0
df_tx.loc[7, "montant"] = 2200.0

print(f"Dataset généré : {df_tx.shape[0]} lignes.")
df_tx.head(5)

### 🧪 Exercice 1 : Déduplication et Imputation ciblée

**Consignes :**
1. Supprimez les doublons basés sur la clé unique `tx_id` en conservant la première occurrence (`drop_duplicates`).
2. Imputez les valeurs manquantes de la colonne `montant` par la **médiane** des montants existants.
3. Imputez les valeurs manquantes de la colonne `statut` par la chaîne `'INCONNU'`.
4. Stockez le résultat assaini dans `df_tx_clean`.

In [ ]:
# TODO : Déduplication et imputation
df_tx_clean = ...

df_tx_clean.head()

In [ ]:
assert len(df_tx_clean) == 1000, f"Attendu 1000 lignes après déduplication, obtenu {len(df_tx_clean)}."
assert df_tx_clean["montant"].isna().sum() == 0, "Il subsiste des NaN dans la colonne montant."
assert df_tx_clean["statut"].isna().sum() == 0, "Il subsiste des NaN dans la colonne statut."
assert "INCONNU" in df_tx_clean["statut"].values, "La catégorie 'INCONNU' n'a pas été insérée."
print("✅ Exercice 1 validé !")

### 🧪 Exercice 2 : Détection d'outliers par écart interquartile (IQR)

**Consignes :**
À partir de `df_tx_clean` :
1. Calculez les quartiles $Q_1$ (25%) et $Q_3$ (75%) de la colonne `montant`.
2. Calculez $\text{IQR} = Q_3 - Q_1$.
3. Déterminez la borne supérieure de Tukey : borne_sup = Q3 + 1.5 * IQR.
4. Isolez les lignes dépassant cette borne dans un DataFrame nommé `df_outliers`.

In [ ]:
# TODO : Calcul de l'IQR et filtrage des outliers
q1 = ...
q3 = ...
iqr = ...
borne_sup = ...

df_outliers = ...
print(f"Borne supérieure IQR : {borne_sup:.2f} €")
print(f"Nombre d'outliers détectés : {len(df_outliers)}")

In [ ]:
assert isinstance(df_outliers, pd.DataFrame), "df_outliers doit être un DataFrame."
assert (df_outliers["montant"] > borne_sup).all(), "Toutes les lignes de df_outliers doivent dépasser borne_sup."
assert len(df_outliers) >= 2, "Les deux montants anormaux injectés (1500 et 2200) doivent être détectés."
print("✅ Exercice 2 validé !")

## Partie 2 : Réconciliation relationnelle (`pd.merge`)

Exécutez la cellule suivante pour créer une table de référence des clients avec leur localisation.

In [ ]:
clients_uniques = [f"CLI-{i}" for i in range(100, 190)]  # Certains clients de 190 à 199 sont absents exprès
villes = ["Lille", "Paris", "Lyon", "Nantes", "Bordeaux"]

df_clients = pd.DataFrame({
    "client_id": clients_uniques,
    "ville": np.random.choice(villes, size=len(clients_uniques)),
    "segment": np.random.choice(["B2B", "B2C"], size=len(clients_uniques))
})
df_clients.head(3)

### 🧪 Exercice 3 : Jointure gauche et audit des orphelins

**Consignes :**
1. Réalisez une jointure gauche (`how='left'`) entre `df_tx_clean` et `df_clients` sur la clé commune `client_id`.
2. Activez le paramètre `indicator=True` pour auditer l'origine des lignes.
3. Stockez le résultat dans `df_merged`.
4. Isolez les transactions orphelines (dont le `client_id` n'existait pas dans la table des clients) dans un DataFrame `df_orphelins`.

In [ ]:
# TODO : Jointure et détection des orphelins
df_merged = ...
df_orphelins = ...

print(f"Total lignes après jointure : {len(df_merged)}")
print(f"Transactions sans client référencé : {len(df_orphelins)}")

In [ ]:
assert len(df_merged) == len(df_tx_clean), "Une jointure gauche ne doit pas modifier le nombre de lignes initial."
assert "_merge" in df_merged.columns, "La colonne d'audit '_merge' doit être présente (indicator=True)."
assert (df_orphelins["_merge"] == "left_only").all(), "Les orphelins doivent avoir la valeur 'left_only'."
print("✅ Exercice 3 validé !")

## Partie 3 : Agrégations avancées & Tableaux croisés

Nettoyons d'abord les transactions orphelines en conservant uniquement les lignes validées avec client existant :

In [ ]:
df_valide = df_merged[df_merged["_merge"] == "both"].drop(columns=["_merge"]).copy()
df_valide.head(3)

### 🧪 Exercice 4 : Multi-agrégation avec `.agg()`

**Consignes :**
À partir de `df_valide` :
1. Effectuez un groupement par `ville` (`groupby('ville', as_index=False)`).
2. Calculez les métriques suivantes via la syntaxe des tuples nommés dans `.agg()` :
   - `ca_total` : somme de `montant`
   - `panier_moyen` : moyenne de `montant` (arrondie à 2 décimales)
   - `nb_transactions` : décompte de `tx_id`
3. Triez le tableau par `ca_total` décroissant et stockez-le dans `df_stats_ville`.

In [ ]:
# TODO : Multi-agrégation nommée
df_stats_ville = ...

df_stats_ville

In [ ]:
assert isinstance(df_stats_ville, pd.DataFrame), "df_stats_ville doit être un DataFrame."
assert set(df_stats_ville.columns) == {"ville", "ca_total", "panier_moyen", "nb_transactions"}, "Vérifiez les noms des colonnes calculées."
assert df_stats_ville["ca_total"].is_monotonic_decreasing, "df_stats_ville doit être trié par ca_total décroissant."
print("✅ Exercice 4 validé !")

### 🧪 Exercice 5 : Tableau croisé dynamique (`pivot_table`)

**Consignes :**
Construisez un tableau croisé dynamique `df_pivot` représentant la **somme des montants** selon deux dimensions :
- En lignes (`index`) : la `ville`
- En colonnes (`columns`) : le `segment` du client (`B2B` vs `B2C`)
- Valeurs agrégées : la colonne `montant`
- Fonction d'agrégation : la somme (`'sum'`)
- Remplacer les cases vides éventuelles par `0.0` (`fill_value=0.0`)

In [ ]:
# TODO : Construction du pivot_table
df_pivot = ...

df_pivot

In [ ]:
assert isinstance(df_pivot, pd.DataFrame), "df_pivot doit être un DataFrame."
assert "B2B" in df_pivot.columns and "B2C" in df_pivot.columns, "Les colonnes du pivot doivent être B2B et B2C."
assert len(df_pivot) == 5, "Il doit y avoir 5 lignes correspondant aux 5 villes."
assert df_pivot.isna().sum().sum() == 0, "Il ne doit pas rester de NaN dans la matrice pivotée."
print("🎉 FÉLICITATIONS ! Vous maîtrisez le nettoyage, les jointures et les agrégations sous Pandas !")